# IEMOCAP & MELD Temporal Data Analysis

Notebook này phân tích dữ liệu hội thoại ở mức dialogue/session/split để xem các tín hiệu thời gian có hỗ trợ idea CIM không.

Trọng tâm:

- duration của utterance
- gap giữa hai lượt nói liên tiếp
- overlap/interruption
- speaker switch
- vị trí lượt nói trong dialogue
- emotion transition giữa các lượt nói

Phần IEMOCAP dùng session-level analysis. Phần MELD dùng train/dev/test và có thêm Season/Episode từ metadata. Notebook không load model/checkpoint.


In [ ]:
from pathlib import Path
import os
import sys
from collections import Counter

CWD = Path.cwd().resolve()
os.environ.setdefault('MPLCONFIGDIR', str(CWD / '.matplotlib_cache'))

import matplotlib
if 'ipykernel' not in sys.modules:
    matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

if (CWD / 'utils').is_dir() and ((CWD / 'data' / 'iemocap').is_dir() or (CWD / 'iemocap').is_dir()):
    PROJECT_ROOT = CWD
elif CWD.name in {'notebook', 'notebooks'} and (CWD.parent / 'utils').is_dir():
    PROJECT_ROOT = CWD.parent
else:
    # Manual fallback: set SER_PROJECT_ROOT before running if your notebook server starts elsewhere.
    PROJECT_ROOT = Path(os.environ.get('SER_PROJECT_ROOT', CWD)).resolve()

assert (PROJECT_ROOT / 'utils').is_dir(), f'Không tìm thấy utils/ từ PROJECT_ROOT={PROJECT_ROOT}'
assert ((PROJECT_ROOT / 'data' / 'iemocap').is_dir() or (PROJECT_ROOT / 'iemocap').is_dir()), f'Không tìm thấy data/iemocap hoặc iemocap/ từ PROJECT_ROOT={PROJECT_ROOT}'

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from utils.iemocap_kaggle import LABEL_NAMES, MELD_RAW_LABEL_MAP, discover_iemocap_samples

IEMOCAP_ROOT = Path(os.environ.get('IEMOCAP_ROOT', PROJECT_ROOT / 'data' / 'iemocap' if (PROJECT_ROOT / 'data' / 'iemocap').is_dir() else PROJECT_ROOT / 'iemocap')).resolve()
REPORT_DIR = PROJECT_ROOT / 'reports' / 'iemocap_temporal_session_analysis'
MELD_REPORT_DIR = PROJECT_ROOT / 'reports' / 'meld_temporal_analysis'
COMPARISON_REPORT_DIR = PROJECT_ROOT / 'reports' / 'temporal_dataset_comparison'
REPORT_DIR.mkdir(parents=True, exist_ok=True)
MELD_REPORT_DIR.mkdir(parents=True, exist_ok=True)
COMPARISON_REPORT_DIR.mkdir(parents=True, exist_ok=True)
MELD_ROOT = Path(os.environ.get('MELD_ROOT', PROJECT_ROOT / 'data' / 'meld-dataset' / 'MELD-RAW' / 'MELD.Raw')).resolve()

print('PROJECT_ROOT =', PROJECT_ROOT)
print('IEMOCAP_ROOT =', IEMOCAP_ROOT)
print('REPORT_DIR   =', REPORT_DIR)
print('MELD_ROOT    =', MELD_ROOT)
print('MELD_REPORT  =', MELD_REPORT_DIR)




## 1. Load IEMOCAP Metadata

Dùng parser hiện có trong repo, nên emotion mapping giống training pipeline hiện tại: 8-class raw label được map về 4 class `angry/happy/neutral/sad`, còn `xxx/oth` bị loại khỏi supervised label.


In [ ]:
samples = discover_iemocap_samples(IEMOCAP_ROOT, auto_download=False)
rows = []
for sample in samples:
    rows.append({
        'session': f'Ses{sample.session_id:02d}',
        'session_id': sample.session_id,
        'dialogue_id': sample.dialogue_id,
        'utterance_id': sample.utterance_id,
        'speaker_id': sample.speaker_id,
        'speaker_role': sample.speaker_id.split('_')[-1],
        'start_time': float(sample.start_time),
        'end_time': float(sample.end_time),
        'emotion': sample.label_name,
        'raw_emotion': sample.raw_label,
        'transcript': sample.transcript,
        'audio_path': sample.audio_path,
    })

base_df = pd.DataFrame(rows)
base_df = base_df.sort_values(['session_id', 'dialogue_id', 'start_time', 'end_time', 'utterance_id']).reset_index(drop=True)
base_df['duration'] = (base_df['end_time'] - base_df['start_time']).clip(lower=0)
print('utterances:', len(base_df))
print('dialogues:', base_df['dialogue_id'].nunique())
print('sessions:', sorted(base_df['session'].unique()))
base_df.head()


## 2. Build Temporal Features

Các feature này tương ứng với nhóm CIM đang dùng: duration, gap, overlap, speaker switch, turn position. Tất cả được tính trong từng dialogue để tránh trộn session/dialogue.


In [ ]:
def add_temporal_features(df: pd.DataFrame) -> pd.DataFrame:
    output = []
    for _, dialogue in df.groupby('dialogue_id', sort=False):
        dialogue = dialogue.sort_values(['start_time', 'end_time', 'utterance_id']).reset_index(drop=True).copy()
        n = len(dialogue)
        prev_end = dialogue['end_time'].shift(1)
        prev_start = dialogue['start_time'].shift(1)
        prev_speaker = dialogue['speaker_id'].shift(1)
        prev_emotion = dialogue['emotion'].shift(1)
        next_start = dialogue['start_time'].shift(-1)

        dialogue['turn_index'] = np.arange(n)
        dialogue['turn_position'] = dialogue['turn_index'] / max(n - 1, 1)
        dialogue['dialogue_len'] = n
        dialogue['gap_prev'] = dialogue['start_time'] - prev_end
        dialogue.loc[dialogue['turn_index'] == 0, 'gap_prev'] = np.nan
        dialogue['gap_next'] = next_start - dialogue['end_time']
        dialogue['overlap_prev'] = (prev_end - dialogue['start_time']).clip(lower=0)
        dialogue.loc[dialogue['turn_index'] == 0, 'overlap_prev'] = 0.0
        dialogue['overlap_ratio'] = dialogue['overlap_prev'] / dialogue['duration'].replace(0, np.nan)
        dialogue['overlap_ratio'] = dialogue['overlap_ratio'].replace([np.inf, -np.inf], np.nan).fillna(0.0).clip(lower=0)
        dialogue['has_overlap'] = dialogue['overlap_prev'] > 0
        dialogue['speaker_switch'] = dialogue['speaker_id'] != prev_speaker
        dialogue.loc[dialogue['turn_index'] == 0, 'speaker_switch'] = False
        dialogue['same_speaker'] = ~dialogue['speaker_switch']
        dialogue.loc[dialogue['turn_index'] == 0, 'same_speaker'] = False
        dialogue['is_interrupting'] = dialogue['has_overlap'] & dialogue['speaker_switch']
        dialogue['prev_emotion'] = prev_emotion
        dialogue['emotion_transition'] = dialogue['prev_emotion'].fillna('START') + ' -> ' + dialogue['emotion']
        dialogue['emotion_changed'] = (dialogue['prev_emotion'].notna()) & (dialogue['prev_emotion'] != dialogue['emotion'])
        dialogue['short_gap'] = dialogue['gap_prev'].between(0, 0.3, inclusive='both')
        dialogue['long_gap'] = dialogue['gap_prev'] > 1.0
        dialogue['early_turn'] = dialogue['turn_position'] <= 0.33
        dialogue['middle_turn'] = dialogue['turn_position'].between(0.33, 0.66, inclusive='right')
        dialogue['late_turn'] = dialogue['turn_position'] > 0.66
        output.append(dialogue)
    return pd.concat(output, ignore_index=True)

df = add_temporal_features(base_df)
df.to_csv(REPORT_DIR / 'iemocap_temporal_features_by_utterance.csv', index=False)
df.head()


## 2.1 Configurable Feature Summary

Sửa `FEATURES_TO_ANALYZE` để chọn danh sách feature cần thống kê theo session, emotion, hoặc session-emotion.
Tên feature có thể là alias như `duration`, `gap_prev`, `overlap`, hoặc tên output trực tiếp như `duration_mean`, `overlap_rate`.


In [ ]:
# User-configurable list. Edit this list before running the cells below.
FEATURES_TO_ANALYZE = [
    "duration",
    "gap_prev",
    "overlap",
    "interruption",
    "speaker_switch",
    "short_gap",
    "long_gap",
    "emotion_change",
    "turn_position",
]

# output_name: (source_column, aggregation, scale)
# scale=100 means rate/proportion is shown as percent.
FEATURE_LIBRARY = {
    "duration_mean": ("duration", "mean", 1.0),
    "duration_median": ("duration", "median", 1.0),
    "gap_prev_mean": ("gap_prev", "mean", 1.0),
    "gap_prev_median": ("gap_prev", "median", 1.0),
    "gap_next_mean": ("gap_next", "mean", 1.0),
    "overlap_prev_mean": ("overlap_prev", "mean", 1.0),
    "overlap_ratio_mean": ("overlap_ratio", "mean", 1.0),
    "overlap_rate": ("has_overlap", "mean", 100.0),
    "interruption_rate": ("is_interrupting", "mean", 100.0),
    "speaker_switch_rate": ("speaker_switch", "mean", 100.0),
    "same_speaker_rate": ("same_speaker", "mean", 100.0),
    "short_gap_rate": ("short_gap", "mean", 100.0),
    "long_gap_rate": ("long_gap", "mean", 100.0),
    "emotion_change_rate": ("emotion_changed", "mean", 100.0),
    "mean_turn_position": ("turn_position", "mean", 1.0),
    "early_turn_rate": ("early_turn", "mean", 100.0),
    "middle_turn_rate": ("middle_turn", "mean", 100.0),
    "late_turn_rate": ("late_turn", "mean", 100.0),
}

FEATURE_ALIASES = {
    "duration": ["duration_mean", "duration_median"],
    "gap": ["gap_prev_mean", "gap_prev_median"],
    "gap_prev": ["gap_prev_mean", "gap_prev_median"],
    "gap_next": ["gap_next_mean"],
    "overlap": ["overlap_rate", "overlap_prev_mean", "overlap_ratio_mean"],
    "overlap_prev": ["overlap_prev_mean"],
    "overlap_ratio": ["overlap_ratio_mean"],
    "interruption": ["interruption_rate"],
    "speaker_switch": ["speaker_switch_rate"],
    "same_speaker": ["same_speaker_rate"],
    "short_gap": ["short_gap_rate"],
    "long_gap": ["long_gap_rate"],
    "emotion_change": ["emotion_change_rate"],
    "turn_position": ["mean_turn_position"],
    "early_turn": ["early_turn_rate"],
    "middle_turn": ["middle_turn_rate"],
    "late_turn": ["late_turn_rate"],
}


def resolve_feature_outputs(features):
    outputs = []
    unknown = []
    for feature in features:
        if feature in FEATURE_ALIASES:
            candidates = FEATURE_ALIASES[feature]
        elif feature in FEATURE_LIBRARY:
            candidates = [feature]
        else:
            candidates = []
            unknown.append(feature)
        for candidate in candidates:
            if candidate not in outputs:
                outputs.append(candidate)
    if unknown:
        available = sorted(set(FEATURE_LIBRARY) | set(FEATURE_ALIASES))
        raise ValueError(f"Unknown feature(s): {unknown}. Available: {available}")
    return outputs


def temporal_summary_by(frame: pd.DataFrame, group_cols, features):
    feature_outputs = resolve_feature_outputs(features)
    named_aggs = {
        "n": ("utterance_id", "count"),
        "dialogues": ("dialogue_id", "nunique"),
    }
    scales = {}
    for output_name in feature_outputs:
        source_col, agg_func, scale = FEATURE_LIBRARY[output_name]
        if source_col not in frame.columns:
            raise ValueError(f"Feature {output_name!r} needs missing source column {source_col!r}.")
        named_aggs[output_name] = (source_col, agg_func)
        scales[output_name] = scale

    summary = frame.groupby(group_cols).agg(**named_aggs).reset_index()
    for output_name, scale in scales.items():
        summary[output_name] = summary[output_name] * scale
    return summary


selected_feature_outputs = resolve_feature_outputs(FEATURES_TO_ANALYZE)
print("Selected feature outputs:", selected_feature_outputs)

selected_features_by_session = temporal_summary_by(df, ["session"], FEATURES_TO_ANALYZE)
selected_features_by_emotion = temporal_summary_by(df, ["emotion"], FEATURES_TO_ANALYZE)
selected_features_by_session_emotion = temporal_summary_by(df, ["session", "emotion"], FEATURES_TO_ANALYZE)

selected_features_by_session.round(3).to_csv(REPORT_DIR / "selected_features_by_session.csv", index=False)
selected_features_by_emotion.round(3).to_csv(REPORT_DIR / "selected_features_by_emotion.csv", index=False)
selected_features_by_session_emotion.round(3).to_csv(REPORT_DIR / "selected_features_by_session_emotion.csv", index=False)

display(selected_features_by_session.round(3))
display(selected_features_by_emotion.round(3))
display(selected_features_by_session_emotion.round(3).head(20))



### Configurable Feature Plots


In [ ]:
def plot_selected_feature_bars(summary: pd.DataFrame, group_col: str, output_path: Path, max_cols: int = 3):
    value_cols = [col for col in summary.columns if col not in {group_col, "n", "dialogues"}]
    if not value_cols:
        print("No selected feature columns to plot.")
        return
    n_cols = min(max_cols, len(value_cols))
    n_rows = int(np.ceil(len(value_cols) / n_cols))
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(5 * n_cols, 3.4 * n_rows), squeeze=False)
    for ax, col in zip(axes.ravel(), value_cols):
        summary.plot(x=group_col, y=col, kind="bar", legend=False, ax=ax, color="#4C78A8")
        ax.set_title(col)
        ax.set_xlabel("")
        ax.grid(axis="y", alpha=0.25)
    for ax in axes.ravel()[len(value_cols):]:
        ax.axis("off")
    plt.tight_layout()
    plt.savefig(output_path, dpi=160)
    plt.show()


plot_selected_feature_bars(selected_features_by_session, "session", REPORT_DIR / "selected_features_by_session.png")
plot_selected_feature_bars(selected_features_by_emotion, "emotion", REPORT_DIR / "selected_features_by_emotion.png")

for feature in selected_feature_outputs[:6]:
    pivot = selected_features_by_session_emotion.pivot(index="session", columns="emotion", values=feature).reindex(columns=LABEL_NAMES)
    fig, ax = plt.subplots(figsize=(8, 4))
    im = ax.imshow(pivot.fillna(0).to_numpy(), aspect="auto", cmap="Blues")
    ax.set_title(f"{feature} by session/emotion")
    ax.set_xticks(range(len(pivot.columns)), pivot.columns)
    ax.set_yticks(range(len(pivot.index)), pivot.index)
    for i in range(pivot.shape[0]):
        for j in range(pivot.shape[1]):
            value = pivot.iloc[i, j]
            text = "" if pd.isna(value) else f"{value:.2f}"
            ax.text(j, i, text, ha="center", va="center", fontsize=8)
    fig.colorbar(im, ax=ax, shrink=0.8)
    plt.tight_layout()
    plt.savefig(REPORT_DIR / f"selected_{feature}_session_emotion_heatmap.png", dpi=160)
    plt.show()



### Configurable Feature Distributions

Phần này dùng cùng `FEATURES_TO_ANALYZE` để xem phân bố utterance-level của từng feature theo `emotion` hoặc `session`.
Feature liên tục được vẽ bằng boxplot; feature nhị phân được vẽ bằng bar rate (%).


In [ ]:
def resolve_distribution_sources(features):
    sources = []
    for output_name in resolve_feature_outputs(features):
        source_col = FEATURE_LIBRARY[output_name][0]
        if source_col not in sources:
            sources.append(source_col)
    return sources


def is_binary_feature(series: pd.Series) -> bool:
    values = series.dropna()
    if values.empty:
        return False
    unique = set(values.astype(float).unique()) if pd.api.types.is_numeric_dtype(values) else set(values.unique())
    return unique.issubset({0, 1, 0.0, 1.0, False, True})


def feature_distribution_stats(frame: pd.DataFrame, group_col: str, features):
    rows = []
    for feature in features:
        if feature not in frame.columns:
            continue
        numeric = pd.to_numeric(frame[feature], errors="coerce").astype(float)
        temp = frame[[group_col]].copy()
        temp[feature] = numeric
        for group_value, group in temp.groupby(group_col, sort=True):
            values = group[feature].dropna()
            if values.empty:
                rows.append({
                    "group_by": group_col,
                    "group": group_value,
                    "feature": feature,
                    "n": 0,
                    "mean": np.nan,
                    "median": np.nan,
                    "std": np.nan,
                    "q25": np.nan,
                    "q75": np.nan,
                    "min": np.nan,
                    "max": np.nan,
                    "rate_percent": np.nan,
                })
                continue
            binary = is_binary_feature(values)
            rows.append({
                "group_by": group_col,
                "group": group_value,
                "feature": feature,
                "n": int(values.shape[0]),
                "mean": float(values.mean()),
                "median": float(values.median()),
                "std": float(values.std(ddof=1)) if values.shape[0] > 1 else 0.0,
                "q25": float(values.quantile(0.25)),
                "q75": float(values.quantile(0.75)),
                "min": float(values.min()),
                "max": float(values.max()),
                "rate_percent": float(values.mean() * 100.0) if binary else np.nan,
            })
    return pd.DataFrame(rows)


def plot_feature_distributions(frame: pd.DataFrame, group_col: str, features, output_path: Path, max_cols: int = 3):
    valid_features = [feature for feature in features if feature in frame.columns]
    if not valid_features:
        print(f"No valid features for {group_col} distribution plot.")
        return
    groups = sorted(frame[group_col].dropna().unique())
    n_cols = min(max_cols, len(valid_features))
    n_rows = int(np.ceil(len(valid_features) / n_cols))
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(5.2 * n_cols, 3.6 * n_rows), squeeze=False)

    for ax, feature in zip(axes.ravel(), valid_features):
        values = pd.to_numeric(frame[feature], errors="coerce").astype(float)
        plot_df = frame[[group_col]].copy()
        plot_df[feature] = values
        binary = is_binary_feature(plot_df[feature])
        if binary:
            rates = plot_df.groupby(group_col)[feature].mean().reindex(groups) * 100.0
            ax.bar(range(len(groups)), rates.values, color="#59A14F")
            ax.set_ylabel("rate (%)")
            ax.set_ylim(0, max(100, np.nanmax(rates.values) * 1.15 if len(rates) else 100))
        else:
            arrays = [plot_df.loc[plot_df[group_col] == group, feature].dropna().values for group in groups]
            ax.boxplot(arrays, labels=groups, showfliers=False)
            ax.set_ylabel(feature)
        ax.set_title(f"{feature} by {group_col}")
        ax.set_xlabel("")
        ax.tick_params(axis="x", rotation=30)
        ax.grid(axis="y", alpha=0.25)

    for ax in axes.ravel()[len(valid_features):]:
        ax.axis("off")
    plt.tight_layout()
    plt.savefig(output_path, dpi=160)
    plt.show()


selected_distribution_features = resolve_distribution_sources(FEATURES_TO_ANALYZE)
print("Selected distribution source columns:", selected_distribution_features)

emotion_distribution_stats = feature_distribution_stats(df, "emotion", selected_distribution_features)
session_distribution_stats = feature_distribution_stats(df, "session", selected_distribution_features)
emotion_distribution_stats.round(4).to_csv(REPORT_DIR / "selected_feature_distribution_stats_by_emotion.csv", index=False)
session_distribution_stats.round(4).to_csv(REPORT_DIR / "selected_feature_distribution_stats_by_session.csv", index=False)

display(emotion_distribution_stats.round(4).head(30))
display(session_distribution_stats.round(4).head(30))

plot_feature_distributions(
    df,
    "emotion",
    selected_distribution_features,
    REPORT_DIR / "selected_feature_distribution_by_emotion.png",
)
plot_feature_distributions(
    df,
    "session",
    selected_distribution_features,
    REPORT_DIR / "selected_feature_distribution_by_session.png",
)




### Low-Redundancy Feature Correlation

Mặc định cell này dùng toàn bộ feature trong heatmap IEMOCAP đã lưu tại `reports/temporal_interaction_study/iemocap/correlation_matrix.csv`, tương ứng với hình `reports/temporal_interaction_study/plots/correlation_heatmaps/iemocap_correlation_heatmap.png`.

Lọc các feature có tương quan với mọi feature khác nằm trong khoảng `[-0.5, 0.5]`, tức `max_abs_corr <= 0.5`.
Nhóm này ít dư thừa hơn và phù hợp để cân nhắc đưa vào CIM/TII.


In [ ]:
CORRELATION_THRESHOLD = 0.5
CORRELATION_METHOD = "spearman"  # Used only when computing from current dataframe.
CORRELATION_FEATURE_SOURCE = "temporal_interaction_study"  # "temporal_interaction_study" or "selected_features"
STUDY_CORRELATION_MATRIX_PATH = PROJECT_ROOT / "reports" / "temporal_interaction_study" / "iemocap" / "correlation_matrix.csv"


def select_low_redundancy_from_corr(corr: pd.DataFrame, threshold: float = 0.5):
    corr = corr.copy()
    corr.index = corr.index.astype(str)
    corr.columns = corr.columns.astype(str)
    shared_features = [feature for feature in corr.columns if feature in corr.index]
    corr = corr.loc[shared_features, shared_features].apply(pd.to_numeric, errors="coerce")

    if corr.empty:
        return [], corr, pd.DataFrame()

    abs_corr = corr.abs().copy()
    diagonal_mask = pd.DataFrame(
        np.eye(len(abs_corr), dtype=bool),
        index=abs_corr.index,
        columns=abs_corr.columns,
    )
    abs_corr = abs_corr.mask(diagonal_mask)
    max_abs_corr = abs_corr.max(axis=1).fillna(0.0)
    selected = max_abs_corr[max_abs_corr <= threshold].sort_values().index.tolist()
    ranking = (
        pd.DataFrame({"feature": max_abs_corr.index, "max_abs_corr_with_other_features": max_abs_corr.values})
        .sort_values("max_abs_corr_with_other_features")
        .reset_index(drop=True)
    )
    ranking["selected_low_redundancy"] = ranking["feature"].isin(selected)
    return selected, corr, ranking


def compute_corr_from_dataframe(frame: pd.DataFrame, features, method: str = "spearman"):
    valid_features = []
    for feature in features:
        if feature not in frame.columns:
            continue
        values = pd.to_numeric(frame[feature], errors="coerce")
        if values.notna().sum() < 3:
            continue
        if values.nunique(dropna=True) <= 1:
            continue
        valid_features.append(feature)
    return frame[valid_features].apply(pd.to_numeric, errors="coerce").corr(method=method)


if CORRELATION_FEATURE_SOURCE == "temporal_interaction_study":
    if not STUDY_CORRELATION_MATRIX_PATH.exists():
        raise FileNotFoundError(f"Missing correlation matrix: {STUDY_CORRELATION_MATRIX_PATH}")
    selected_feature_corr = pd.read_csv(STUDY_CORRELATION_MATRIX_PATH, index_col=0)
    corr_candidate_features = selected_feature_corr.columns.astype(str).tolist()
    output_prefix = "temporal_study"
    source_note = str(STUDY_CORRELATION_MATRIX_PATH.relative_to(PROJECT_ROOT))
else:
    corr_candidate_features = selected_distribution_features if "selected_distribution_features" in globals() else resolve_distribution_sources(FEATURES_TO_ANALYZE)
    selected_feature_corr = compute_corr_from_dataframe(df, corr_candidate_features, method=CORRELATION_METHOD)
    corr_candidate_features = selected_feature_corr.columns.astype(str).tolist()
    output_prefix = "selected_feature"
    source_note = f"current dataframe / {CORRELATION_METHOD}"

low_corr_features, selected_feature_corr, low_corr_ranking = select_low_redundancy_from_corr(
    selected_feature_corr,
    threshold=CORRELATION_THRESHOLD,
)

low_corr_ranking.to_csv(REPORT_DIR / f"{output_prefix}_correlation_redundancy_ranking.csv", index=False)
selected_feature_corr.to_csv(REPORT_DIR / f"{output_prefix}_full_correlation_matrix.csv")
selected_feature_corr.loc[low_corr_features, low_corr_features].to_csv(REPORT_DIR / f"{output_prefix}_low_redundancy_feature_correlation_matrix.csv")

print(f"Correlation source: {source_note}")
print(f"Threshold: max_abs_corr <= {CORRELATION_THRESHOLD}")
print(f"Candidate features: {len(corr_candidate_features)}")
print(f"Low-redundancy selected features: {len(low_corr_features)}")
print(low_corr_features)
display(low_corr_ranking)

if len(low_corr_features) >= 2:
    low_corr_matrix = selected_feature_corr.loc[low_corr_features, low_corr_features]
    fig, ax = plt.subplots(figsize=(max(6, 0.8 * len(low_corr_features)), max(5, 0.7 * len(low_corr_features))))
    im = ax.imshow(low_corr_matrix.to_numpy(), vmin=-1, vmax=1, cmap="coolwarm", aspect="auto")
    ax.set_title(f"Low-redundancy feature correlation ({output_prefix})")
    ax.set_xticks(range(len(low_corr_features)), low_corr_features, rotation=45, ha="right")
    ax.set_yticks(range(len(low_corr_features)), low_corr_features)
    for i in range(low_corr_matrix.shape[0]):
        for j in range(low_corr_matrix.shape[1]):
            ax.text(j, i, f"{low_corr_matrix.iloc[i, j]:.2f}", ha="center", va="center", fontsize=8)
    fig.colorbar(im, ax=ax, shrink=0.85)
    plt.tight_layout()
    plt.savefig(REPORT_DIR / f"{output_prefix}_low_redundancy_feature_correlation.png", dpi=180)
    plt.show()
else:
    print("Not enough low-redundancy features to draw a separate correlation heatmap.")

if len(corr_candidate_features) >= 2:
    full_corr = selected_feature_corr.loc[corr_candidate_features, corr_candidate_features]
    fig, ax = plt.subplots(figsize=(max(7, 0.75 * len(corr_candidate_features)), max(6, 0.65 * len(corr_candidate_features))))
    im = ax.imshow(full_corr.to_numpy(), vmin=-1, vmax=1, cmap="coolwarm", aspect="auto")
    ax.set_title(f"Full temporal feature correlation ({output_prefix})")
    ax.set_xticks(range(len(corr_candidate_features)), corr_candidate_features, rotation=45, ha="right")
    ax.set_yticks(range(len(corr_candidate_features)), corr_candidate_features)
    fig.colorbar(im, ax=ax, shrink=0.85)
    plt.tight_layout()
    plt.savefig(REPORT_DIR / f"{output_prefix}_full_feature_correlation.png", dpi=180)
    plt.show()


## 3. Session-Level Temporal Summary

Bảng này trả lời câu hỏi: các session có khác nhau rõ về nhịp hội thoại không? Nếu có, CIM có cơ hội học thêm thông tin ngoài acoustic embedding.


In [ ]:
def pct(series):
    return float(series.mean()) if len(series) else 0.0

session_summary = (
    df.groupby('session')
    .agg(
        utterances=('utterance_id', 'count'),
        dialogues=('dialogue_id', 'nunique'),
        mean_dialogue_len=('dialogue_len', 'mean'),
        duration_mean=('duration', 'mean'),
        duration_median=('duration', 'median'),
        gap_prev_mean=('gap_prev', 'mean'),
        gap_prev_median=('gap_prev', 'median'),
        overlap_rate=('has_overlap', pct),
        interruption_rate=('is_interrupting', pct),
        speaker_switch_rate=('speaker_switch', pct),
        short_gap_rate=('short_gap', pct),
        long_gap_rate=('long_gap', pct),
        emotion_change_rate=('emotion_changed', pct),
    )
    .reset_index()
)

for col in ['overlap_rate', 'interruption_rate', 'speaker_switch_rate', 'short_gap_rate', 'long_gap_rate', 'emotion_change_rate']:
    session_summary[col] = session_summary[col] * 100

session_summary.round(3).to_csv(REPORT_DIR / 'session_temporal_summary.csv', index=False)
session_summary.round(3)


In [ ]:
plot_cols = ['duration_mean', 'gap_prev_mean', 'overlap_rate', 'interruption_rate', 'speaker_switch_rate', 'emotion_change_rate']
fig, axes = plt.subplots(2, 3, figsize=(14, 7))
for ax, col in zip(axes.ravel(), plot_cols):
    session_summary.plot(x='session', y=col, kind='bar', legend=False, ax=ax, color='#4C78A8')
    ax.set_title(col)
    ax.set_xlabel('')
    ax.grid(axis='y', alpha=0.25)
plt.tight_layout()
plt.savefig(REPORT_DIR / 'session_temporal_summary.png', dpi=160)
plt.show()


## 4. Emotion Distribution Per Session

Nếu một session vừa có temporal pattern khác, vừa có emotion distribution khác, CIM có thể tận dụng hoặc bị bias bởi pattern đó. Đây là chỗ cần nhìn khi Session 5 outlier.


In [ ]:
emotion_session = pd.crosstab(df['session'], df['emotion'])
emotion_session_ratio = pd.crosstab(df['session'], df['emotion'], normalize='index') * 100
emotion_session.to_csv(REPORT_DIR / 'session_emotion_counts.csv')
emotion_session_ratio.round(2).to_csv(REPORT_DIR / 'session_emotion_ratio.csv')
print('Counts')
display(emotion_session)
print('Ratio (%)')
display(emotion_session_ratio.round(2))


In [ ]:
ax = emotion_session_ratio[LABEL_NAMES].plot(kind='bar', stacked=True, figsize=(10, 4), colormap='tab20')
ax.set_title('Emotion ratio by session')
ax.set_ylabel('% utterances')
ax.set_xlabel('session')
ax.legend(title='emotion', bbox_to_anchor=(1.02, 1), loc='upper left')
plt.tight_layout()
plt.savefig(REPORT_DIR / 'session_emotion_ratio.png', dpi=160)
plt.show()


## 5. Temporal Features By Emotion

Nếu temporal features khác nhau theo emotion, đây là bằng chứng ủng hộ CIM. Ví dụ angry có interruption/overlap cao hơn, sad có duration/gap dài hơn, neutral nằm nhiều ở vùng ít overlap.


In [ ]:
emotion_temporal = (
    df.groupby('emotion')
    .agg(
        n=('utterance_id', 'count'),
        duration_mean=('duration', 'mean'),
        duration_median=('duration', 'median'),
        gap_prev_mean=('gap_prev', 'mean'),
        gap_prev_median=('gap_prev', 'median'),
        overlap_rate=('has_overlap', pct),
        interruption_rate=('is_interrupting', pct),
        speaker_switch_rate=('speaker_switch', pct),
        short_gap_rate=('short_gap', pct),
        long_gap_rate=('long_gap', pct),
        mean_turn_position=('turn_position', 'mean'),
    )
    .reset_index()
)
for col in ['overlap_rate', 'interruption_rate', 'speaker_switch_rate', 'short_gap_rate', 'long_gap_rate']:
    emotion_temporal[col] = emotion_temporal[col] * 100
emotion_temporal.round(3).to_csv(REPORT_DIR / 'emotion_temporal_summary.csv', index=False)
emotion_temporal.round(3)


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, col in zip(axes, ['duration_mean', 'overlap_rate', 'long_gap_rate']):
    emotion_temporal.plot(x='emotion', y=col, kind='bar', legend=False, ax=ax, color='#59A14F')
    ax.set_title(col)
    ax.set_xlabel('')
    ax.grid(axis='y', alpha=0.25)
plt.tight_layout()
plt.savefig(REPORT_DIR / 'emotion_temporal_summary.png', dpi=160)
plt.show()


## 6. Session x Emotion Temporal Summary

Bảng này quan trọng để xem CIM có đang học tín hiệu ổn định hay chỉ học bias của một session. Nếu overlap chỉ mạnh ở một session/emotion cụ thể, nó có thể gây nhiễu khi LOSO.


In [ ]:
session_emotion_temporal = (
    df.groupby(['session', 'emotion'])
    .agg(
        n=('utterance_id', 'count'),
        duration_mean=('duration', 'mean'),
        gap_prev_mean=('gap_prev', 'mean'),
        overlap_rate=('has_overlap', pct),
        interruption_rate=('is_interrupting', pct),
        speaker_switch_rate=('speaker_switch', pct),
        long_gap_rate=('long_gap', pct),
    )
    .reset_index()
)
for col in ['overlap_rate', 'interruption_rate', 'speaker_switch_rate', 'long_gap_rate']:
    session_emotion_temporal[col] = session_emotion_temporal[col] * 100
session_emotion_temporal.round(3).to_csv(REPORT_DIR / 'session_emotion_temporal_summary.csv', index=False)
session_emotion_temporal.round(3)


In [ ]:
pivot_overlap = session_emotion_temporal.pivot(index='session', columns='emotion', values='overlap_rate').reindex(columns=LABEL_NAMES)
pivot_duration = session_emotion_temporal.pivot(index='session', columns='emotion', values='duration_mean').reindex(columns=LABEL_NAMES)
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].imshow(pivot_overlap.fillna(0).to_numpy(), aspect='auto', cmap='Blues')
axes[0].set_title('Overlap rate (%) by session/emotion')
axes[0].set_xticks(range(len(pivot_overlap.columns)), pivot_overlap.columns)
axes[0].set_yticks(range(len(pivot_overlap.index)), pivot_overlap.index)
for i in range(pivot_overlap.shape[0]):
    for j in range(pivot_overlap.shape[1]):
        axes[0].text(j, i, f'{pivot_overlap.iloc[i, j]:.1f}', ha='center', va='center', fontsize=8)

axes[1].imshow(pivot_duration.fillna(0).to_numpy(), aspect='auto', cmap='Greens')
axes[1].set_title('Mean duration by session/emotion')
axes[1].set_xticks(range(len(pivot_duration.columns)), pivot_duration.columns)
axes[1].set_yticks(range(len(pivot_duration.index)), pivot_duration.index)
for i in range(pivot_duration.shape[0]):
    for j in range(pivot_duration.shape[1]):
        axes[1].text(j, i, f'{pivot_duration.iloc[i, j]:.2f}', ha='center', va='center', fontsize=8)
plt.tight_layout()
plt.savefig(REPORT_DIR / 'session_emotion_temporal_heatmaps.png', dpi=160)
plt.show()


## 7. Emotion Transitions

CIM có thể hữu ích nếu emotion phụ thuộc vào nhịp chuyển lượt và trạng thái trước đó, ví dụ `neutral -> angry`, `angry -> angry`, hoặc các đoạn có interruption.


In [ ]:
transition_counts = (
    df[df['prev_emotion'].notna()]
    .groupby(['prev_emotion', 'emotion'])
    .size()
    .rename('count')
    .reset_index()
    .sort_values('count', ascending=False)
)
transition_counts.to_csv(REPORT_DIR / 'emotion_transition_counts.csv', index=False)
transition_counts.head(20)


In [ ]:
transition_temporal = (
    df[df['prev_emotion'].notna()]
    .groupby(['prev_emotion', 'emotion'])
    .agg(
        n=('utterance_id', 'count'),
        duration_mean=('duration', 'mean'),
        gap_prev_mean=('gap_prev', 'mean'),
        overlap_rate=('has_overlap', pct),
        interruption_rate=('is_interrupting', pct),
        speaker_switch_rate=('speaker_switch', pct),
    )
    .reset_index()
)
for col in ['overlap_rate', 'interruption_rate', 'speaker_switch_rate']:
    transition_temporal[col] = transition_temporal[col] * 100
transition_temporal = transition_temporal.sort_values(['n', 'overlap_rate'], ascending=[False, False])
transition_temporal.round(3).to_csv(REPORT_DIR / 'emotion_transition_temporal_summary.csv', index=False)
transition_temporal.head(20).round(3)


## 8. Quick Evidence For/Against CIM

Cell này tự tạo một bảng kết luận ngắn. Đây không phải kết quả model, nhưng giúp quyết định temporal cue nào đáng giữ hoặc cần ablation.


In [ ]:
evidence_rows = []

# Feature variability across emotions: large range means feature can carry emotion information.
for feature in ['duration_mean', 'gap_prev_mean', 'overlap_rate', 'interruption_rate', 'speaker_switch_rate', 'long_gap_rate']:
    values = emotion_temporal.set_index('emotion')[feature].dropna()
    evidence_rows.append({
        'question': 'emotion_separability',
        'feature': feature,
        'range_across_emotions': float(values.max() - values.min()),
        'max_emotion': values.idxmax(),
        'min_emotion': values.idxmin(),
    })

# Session instability: large range means cue may be useful but also risky under LOSO.
for feature in ['duration_mean', 'gap_prev_mean', 'overlap_rate', 'interruption_rate', 'speaker_switch_rate', 'emotion_change_rate']:
    values = session_summary.set_index('session')[feature].dropna()
    evidence_rows.append({
        'question': 'session_instability',
        'feature': feature,
        'range_across_sessions': float(values.max() - values.min()),
        'max_session': values.idxmax(),
        'min_session': values.idxmin(),
    })

evidence_df = pd.DataFrame(evidence_rows)
evidence_df.to_csv(REPORT_DIR / 'cim_interaction_evidence_summary.csv', index=False)
evidence_df


In [ ]:
print('Saved reports to:', REPORT_DIR)
for path in sorted(REPORT_DIR.glob('*')):
    print('-', path.name)


## 9. Load MELD Metadata

MELD có `StartTime` và `EndTime` trong các file `*_sent_emo.csv`. Khác IEMOCAP, MELD là multi-party và có `Season`, `Episode`, `Dialogue_ID`, `Utterance_ID`. Ở đây ta map emotion 7-class của MELD về 4-class giống pipeline hiện tại để phân tích CIM nhất quán.


In [ ]:
MELD_CSV_PATHS = {
    'train': MELD_ROOT / 'train' / 'train_sent_emo.csv',
    'dev': MELD_ROOT / 'dev_sent_emo.csv',
    'test': MELD_ROOT / 'test_sent_emo.csv',
}

def parse_meld_time(value) -> float:
    text = str(value).strip().replace(',', '.')
    parts = text.split(':')
    if len(parts) != 3:
        return np.nan
    hours, minutes, seconds = parts
    return int(hours) * 3600.0 + int(minutes) * 60.0 + float(seconds)

meld_rows = []
for split, csv_path in MELD_CSV_PATHS.items():
    assert csv_path.is_file(), f'Missing MELD CSV: {csv_path}'
    split_df = pd.read_csv(csv_path)
    for _, row in split_df.iterrows():
        raw_emotion = str(row['Emotion']).lower()
        mapped_emotion = MELD_RAW_LABEL_MAP.get(raw_emotion)
        if mapped_emotion is None:
            continue
        dialogue_id = f"MELD_{split}_dia{int(row['Dialogue_ID'])}"
        utterance_id = f"{dialogue_id}_utt{int(row['Utterance_ID'])}"
        meld_rows.append({
            'dataset': 'MELD',
            'split': split,
            'session': split,
            'season': int(row['Season']),
            'episode': int(row['Episode']),
            'dialogue_id': dialogue_id,
            'utterance_id': utterance_id,
            'dialogue_index': int(row['Dialogue_ID']),
            'utterance_index': int(row['Utterance_ID']),
            'speaker_id': str(row['Speaker']),
            'start_time': parse_meld_time(row['StartTime']),
            'end_time': parse_meld_time(row['EndTime']),
            'emotion': mapped_emotion,
            'raw_emotion': raw_emotion,
            'sentiment': row['Sentiment'],
            'transcript': row['Utterance'],
        })

meld_base_df = pd.DataFrame(meld_rows)
meld_base_df = meld_base_df.dropna(subset=['start_time', 'end_time']).copy()
meld_base_df = meld_base_df.sort_values(['split', 'dialogue_index', 'utterance_index']).reset_index(drop=True)
meld_base_df['duration'] = (meld_base_df['end_time'] - meld_base_df['start_time']).clip(lower=0)
meld_base_df.to_csv(MELD_REPORT_DIR / 'meld_metadata_4class.csv', index=False)

print('MELD utterances:', len(meld_base_df))
print('MELD dialogues:', meld_base_df['dialogue_id'].nunique())
print('splits:', meld_base_df['split'].value_counts().to_dict())
print('raw emotions:', meld_base_df['raw_emotion'].value_counts().to_dict())
meld_base_df.head()


## 10. MELD Temporal Features

Dùng cùng hàm `add_temporal_features()` với IEMOCAP. Với MELD, overlap được kỳ vọng thấp hơn vì timestamp đến từ subtitle/clip segmentation, nên gap/duration/speaker dynamics có thể quan trọng hơn overlap.


In [ ]:
meld_df = add_temporal_features(meld_base_df)
meld_df.to_csv(MELD_REPORT_DIR / 'meld_temporal_features_by_utterance.csv', index=False)

meld_split_summary = (
    meld_df.groupby('split')
    .agg(
        utterances=('utterance_id', 'count'),
        dialogues=('dialogue_id', 'nunique'),
        seasons=('season', 'nunique'),
        episodes=('episode', 'nunique'),
        speakers=('speaker_id', 'nunique'),
        mean_dialogue_len=('dialogue_len', 'mean'),
        duration_mean=('duration', 'mean'),
        duration_median=('duration', 'median'),
        gap_prev_mean=('gap_prev', 'mean'),
        gap_prev_median=('gap_prev', 'median'),
        overlap_rate=('has_overlap', pct),
        interruption_rate=('is_interrupting', pct),
        speaker_switch_rate=('speaker_switch', pct),
        short_gap_rate=('short_gap', pct),
        long_gap_rate=('long_gap', pct),
        emotion_change_rate=('emotion_changed', pct),
    )
    .reset_index()
)
for col in ['overlap_rate', 'interruption_rate', 'speaker_switch_rate', 'short_gap_rate', 'long_gap_rate', 'emotion_change_rate']:
    meld_split_summary[col] = meld_split_summary[col] * 100
meld_split_summary.round(3).to_csv(MELD_REPORT_DIR / 'meld_split_temporal_summary.csv', index=False)
meld_split_summary.round(3)


In [ ]:
meld_emotion_summary = (
    meld_df.groupby('emotion')
    .agg(
        n=('utterance_id', 'count'),
        duration_mean=('duration', 'mean'),
        duration_median=('duration', 'median'),
        gap_prev_mean=('gap_prev', 'mean'),
        gap_prev_median=('gap_prev', 'median'),
        overlap_rate=('has_overlap', pct),
        interruption_rate=('is_interrupting', pct),
        speaker_switch_rate=('speaker_switch', pct),
        short_gap_rate=('short_gap', pct),
        long_gap_rate=('long_gap', pct),
        mean_turn_position=('turn_position', 'mean'),
    )
    .reset_index()
)
for col in ['overlap_rate', 'interruption_rate', 'speaker_switch_rate', 'short_gap_rate', 'long_gap_rate']:
    meld_emotion_summary[col] = meld_emotion_summary[col] * 100
meld_emotion_summary.round(3).to_csv(MELD_REPORT_DIR / 'meld_emotion_temporal_summary.csv', index=False)
meld_emotion_summary.round(3)


In [ ]:
meld_season_summary = (
    meld_df.groupby('season')
    .agg(
        utterances=('utterance_id', 'count'),
        dialogues=('dialogue_id', 'nunique'),
        episodes=('episode', 'nunique'),
        speakers=('speaker_id', 'nunique'),
        duration_mean=('duration', 'mean'),
        gap_prev_mean=('gap_prev', 'mean'),
        overlap_rate=('has_overlap', pct),
        speaker_switch_rate=('speaker_switch', pct),
        emotion_change_rate=('emotion_changed', pct),
    )
    .reset_index()
)
for col in ['overlap_rate', 'speaker_switch_rate', 'emotion_change_rate']:
    meld_season_summary[col] = meld_season_summary[col] * 100
meld_season_summary.round(3).to_csv(MELD_REPORT_DIR / 'meld_season_temporal_summary.csv', index=False)
meld_season_summary.round(3)


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, col in zip(axes, ['duration_mean', 'gap_prev_mean', 'overlap_rate']):
    meld_emotion_summary.plot(x='emotion', y=col, kind='bar', legend=False, ax=ax, color='#F28E2B')
    ax.set_title(f'MELD {col}')
    ax.set_xlabel('')
    ax.grid(axis='y', alpha=0.25)
plt.tight_layout()
plt.savefig(MELD_REPORT_DIR / 'meld_emotion_temporal_summary.png', dpi=160)
plt.show()

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, col in zip(axes, ['duration_mean', 'gap_prev_mean', 'overlap_rate']):
    meld_split_summary.plot(x='split', y=col, kind='bar', legend=False, ax=ax, color='#B07AA1')
    ax.set_title(f'MELD split {col}')
    ax.set_xlabel('')
    ax.grid(axis='y', alpha=0.25)
plt.tight_layout()
plt.savefig(MELD_REPORT_DIR / 'meld_split_temporal_summary.png', dpi=160)
plt.show()


## 11. MELD Emotion Transitions

MELD là multi-party nên transition còn chịu tác động của speaker re-entry và speaker dominance. Bảng dưới giúp xem các transition nào đi kèm gap/overlap/speaker switch mạnh.


In [ ]:
meld_transition_temporal = (
    meld_df[meld_df['prev_emotion'].notna()]
    .groupby(['prev_emotion', 'emotion'])
    .agg(
        n=('utterance_id', 'count'),
        duration_mean=('duration', 'mean'),
        gap_prev_mean=('gap_prev', 'mean'),
        overlap_rate=('has_overlap', pct),
        interruption_rate=('is_interrupting', pct),
        speaker_switch_rate=('speaker_switch', pct),
    )
    .reset_index()
)
for col in ['overlap_rate', 'interruption_rate', 'speaker_switch_rate']:
    meld_transition_temporal[col] = meld_transition_temporal[col] * 100
meld_transition_temporal = meld_transition_temporal.sort_values(['n', 'overlap_rate'], ascending=[False, False])
meld_transition_temporal.round(3).to_csv(MELD_REPORT_DIR / 'meld_emotion_transition_temporal_summary.csv', index=False)
meld_transition_temporal.head(20).round(3)


## 12. IEMOCAP vs MELD Temporal Comparison

So sánh này giúp quyết định feature nào có khả năng generalize. Nếu một feature rất mạnh ở IEMOCAP nhưng yếu ở MELD, nó nên được kiểm soát bằng ablation/gating thay vì hard-code trọng số cao.


In [ ]:
iemocap_compare = df.assign(dataset='IEMOCAP', split_or_session=df['session'])
meld_compare = meld_df.assign(dataset='MELD', split_or_session=meld_df['split'])
combined = pd.concat([iemocap_compare, meld_compare], ignore_index=True, sort=False)

comparison = (
    combined.groupby('dataset')
    .agg(
        utterances=('utterance_id', 'count'),
        dialogues=('dialogue_id', 'nunique'),
        duration_mean=('duration', 'mean'),
        duration_median=('duration', 'median'),
        gap_prev_mean=('gap_prev', 'mean'),
        gap_prev_median=('gap_prev', 'median'),
        overlap_rate=('has_overlap', pct),
        interruption_rate=('is_interrupting', pct),
        speaker_switch_rate=('speaker_switch', pct),
        short_gap_rate=('short_gap', pct),
        long_gap_rate=('long_gap', pct),
        emotion_change_rate=('emotion_changed', pct),
    )
    .reset_index()
)
for col in ['overlap_rate', 'interruption_rate', 'speaker_switch_rate', 'short_gap_rate', 'long_gap_rate', 'emotion_change_rate']:
    comparison[col] = comparison[col] * 100
comparison.round(3).to_csv(COMPARISON_REPORT_DIR / 'iemocap_vs_meld_temporal_summary.csv', index=False)
comparison.round(3)


In [ ]:
comparison_by_emotion = (
    combined.groupby(['dataset', 'emotion'])
    .agg(
        n=('utterance_id', 'count'),
        duration_mean=('duration', 'mean'),
        gap_prev_mean=('gap_prev', 'mean'),
        overlap_rate=('has_overlap', pct),
        speaker_switch_rate=('speaker_switch', pct),
        long_gap_rate=('long_gap', pct),
    )
    .reset_index()
)
for col in ['overlap_rate', 'speaker_switch_rate', 'long_gap_rate']:
    comparison_by_emotion[col] = comparison_by_emotion[col] * 100
comparison_by_emotion.round(3).to_csv(COMPARISON_REPORT_DIR / 'iemocap_vs_meld_by_emotion.csv', index=False)
comparison_by_emotion.round(3)


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, col in zip(axes, ['duration_mean', 'gap_prev_mean', 'overlap_rate']):
    comparison.plot(x='dataset', y=col, kind='bar', legend=False, ax=ax, color='#76B7B2')
    ax.set_title(col)
    ax.set_xlabel('')
    ax.grid(axis='y', alpha=0.25)
plt.tight_layout()
plt.savefig(COMPARISON_REPORT_DIR / 'iemocap_vs_meld_temporal_summary.png', dpi=160)
plt.show()


In [ ]:
meld_evidence_rows = []
for feature in ['duration_mean', 'gap_prev_mean', 'overlap_rate', 'interruption_rate', 'speaker_switch_rate', 'long_gap_rate']:
    values = meld_emotion_summary.set_index('emotion')[feature].dropna()
    meld_evidence_rows.append({
        'dataset': 'MELD',
        'question': 'emotion_separability',
        'feature': feature,
        'range_across_emotions': float(values.max() - values.min()),
        'max_emotion': values.idxmax(),
        'min_emotion': values.idxmin(),
    })
for feature in ['duration_mean', 'gap_prev_mean', 'overlap_rate', 'speaker_switch_rate', 'emotion_change_rate']:
    values = meld_split_summary.set_index('split')[feature].dropna()
    meld_evidence_rows.append({
        'dataset': 'MELD',
        'question': 'split_instability',
        'feature': feature,
        'range_across_splits': float(values.max() - values.min()),
        'max_split': values.idxmax(),
        'min_split': values.idxmin(),
    })
meld_evidence = pd.DataFrame(meld_evidence_rows)
meld_evidence.to_csv(MELD_REPORT_DIR / 'meld_cim_interaction_evidence_summary.csv', index=False)
meld_evidence


In [ ]:
print('IEMOCAP reports:', REPORT_DIR)
for path in sorted(REPORT_DIR.glob('*')):
    print('  -', path.name)
print()
print('MELD reports:', MELD_REPORT_DIR)
for path in sorted(MELD_REPORT_DIR.glob('*')):
    print('  -', path.name)
print()
print('Comparison reports:', COMPARISON_REPORT_DIR)
for path in sorted(COMPARISON_REPORT_DIR.glob('*')):
    print('  -', path.name)
